# 0910 9일차

## 0. 파이썬 문법: 판다스 → 넘파이

산탄데르에서 y가 `Series`(판다스)로 나옴. 넘파이로 바꾸는 방법이 세 가지

```python
y = np.array(y)     # 방법 1 - 넘파이 함수로 감싸기
y = y.to_numpy()    # 방법 2 - 판다스 메서드
y = y.values        # 방법 3 - 판다스 속성
```

| | 형태 | 비고 |
|---|---|---|
| `np.array(y)` | 함수 | 리스트·튜플에도 씀 |
| `y.to_numpy()` | **메서드** (괄호 O) | 판다스가 권장하는 쪽 |
| `y.values` | **속성** (괄호 X) | 8일차에서 원핫 뒤에 붙이던 그것 |

→ `.values`에 괄호가 없는 건 8일차 §0의 메서드 체이닝과 같은 이야기. **속성이라 호출하는 게 아님**

→ 7일차 §0의 "속성 접근 `.` vs 키 접근 `[]`"과 연결됨

### 바꾸는 이유 - 넘파이로 통일

```python
y = pd.get_dummies(y).values    # get_dummies는 DataFrame을 돌려주므로 .values로 꺼냄
```

→ 원핫 결과가 판다스면 `np.argmax(y_test, axis=1)` 같은 넘파이 연산에서 축·인덱스가 헷갈림

→ 케라스도 내부에서는 넘파이로 바꿔 쓰므로, **처음부터 넘파이로 통일**해두는 게 안전

### cf) `get_dummies`에 `dtype=float`을 빠뜨린 것

```python
y = pd.get_dummies(y).values                # keras24 - dtype 지정 안 함 -> bool
y = pd.get_dummies(y, dtype=float).values   # keras26, 28 - float
```

→ 8일차 §3에 "기본 dtype이 `bool`"이라고 정리해뒀는데 keras24에서 빠뜨림

→ 돌아가기는 함(텐서플로가 캐스팅해줌). 그래도 **`dtype=float`을 붙이는 습관**을 들일 것

## 1. 이진분류를 다중분류로 - 산탄데르

산탄데르는 y가 0/1 두 개 → 원래는 7일차에서 배운 이진분류

이걸 **라벨 2개짜리 다중분류**로 바꿔서 풀어봄 (`keras24`)

| | 이진으로 (7일차) | 다중으로 (오늘) |
|---|---|---|
| y 가공 | 없음 `(n,)` | **원핫** `(n, 2)` |
| 출력층 | `Dense(1, activation='sigmoid')` | `Dense(2, activation='softmax')` |
| loss | `binary_crossentropy` | `categorical_crossentropy` |
| 예측 후처리 | `np.round` | `np.argmax` |

→ **라벨이 2개면 둘 다 됨.** 이진분류는 다중분류의 특수한 경우라고 볼 수 있음

→ 보통은 이진 쪽이 간단함. 노드가 1개라 파라미터도 적고 원핫도 필요 없음

### 제출 - 확률 열 고르기 `[:, 1]`

sigmoid는 값이 하나라 그대로 내면 되는데, softmax는 `(n, 2)`라 **어느 열을 낼지 정해야 함**

```python
y_submit = model.predict(test)      # (200000, 2)  [[0.97, 0.03], [0.88, 0.12], ...]
submit['target'] = y_submit[:, 1]   # 1번(양성) 열만
```

```
    0열      1열
[[ 0.97 ,  0.03 ]]     <- 0일 확률 / 1일 확률
           ^^^^
           이 열이 제출값
```

→ 대회가 요구하는 건 "target이 1일 확률". 그러니 **1번 열**

→ 여기서 `argmax`를 쓰면 0 또는 1만 남아 확률 정보가 사라짐. **채점용은 argmax, 제출용은 확률 그대로**

### acc 0.91 - 베이스라인이 0.8995

```
acc_score :  0.91105
time :  300.07 sec
```

라벨 분포를 보면

```
0 : 179902
1 :  20098      <- 10%
```

→ **전부 0이라고만 찍어도 정확도 0.89951**

→ 0.911은 거기서 1.2%p 높은 값 (학습 300초)

→ 7일차 §3의 클래스 불균형. `stratify=y`로 비율은 지켰지만 **불균형 자체가 사라지진 않음**

→ 이런 데이터에서 accuracy로는 성능을 판단하기 어려움 (산탄데르 대회의 실제 평가지표도 AUC)

## 2. `summary()`로 파라미터 개수 세기

```python
model.summary()
```

모델이 학습할 **w와 b가 총 몇 개인지** 보여줌 (`keras25`)

```
┌─────────────────┬──────────────┬───────────┐
│ Layer (type)    │ Output Shape │   Param # │
├─────────────────┼──────────────┼───────────┤
│ dense (Dense)   │ (None, 3)    │         6 │
│ dense_1 (Dense) │ (None, 4)    │        16 │
│ dense_2 (Dense) │ (None, 3)    │        15 │
│ dense_3 (Dense) │ (None, 1)    │         4 │
└─────────────────┴──────────────┴───────────┘
 Total params: 41
```

### 공식 - (입력 + 1) × 출력

$$\text{Param} = (\text{입력 노드 수} + 1) \times \text{출력 노드 수}$$

**+1이 bias**. 출력 노드마다 b가 하나씩 붙음

| 층 | 입력 | 출력 | 계산 | Param |
|---|---|---|---|---|
| `Dense(3, input_dim=1)` | 1 | 3 | (1+1) × 3 | **6** |
| `Dense(4)` | 3 | 4 | (3+1) × 4 | **16** |
| `Dense(3)` | 4 | 3 | (4+1) × 3 | **15** |
| `Dense(1)` | 3 | 1 | (3+1) × 1 | **4** |
| | | | | **41** |

→ 1일차 §3의 `y = wx + b`가 노드 하나마다 하나씩 있는 것

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(3, input_dim=1))    # (1+1) * 3 = 6
model.add(Dense(4))                 # (3+1) * 4 = 16
model.add(Dense(3))                 # (4+1) * 3 = 15
model.add(Dense(1))                 # (3+1) * 1 = 4

model.summary()                     # Total params: 41

## 3. `input_dim` 대신 `input_shape`

```python
model.add(Dense(10, input_dim=4, activation='relu'))        # 지금까지
model.add(Dense(10, input_shape=(4,), activation='relu'))   # 오늘
```

→ 둘은 **같은 말**. iris는 열이 4개니까 `4` 또는 `(4,)`

→ 그런데 `input_dim`은 **숫자 하나**라서 1차원(열 개수)밖에 표현하지 못함

### 규칙 - 맨 앞 행 개수를 떼고 튜플로

```
data shape           input shape
(n, 4)             →  (4,)
(n, 100, 3)        →  (100, 3)
(n, 100, 100, 3)   →  (100, 100, 3)
```

→ 행 개수 `n`은 **데이터가 몇 개인지**일 뿐, 모델 구조와 무관해서 뺌

→ `summary()`의 `Output Shape`가 `(None, 3)`인 것도 같은 이유. **`None`이 그 `n` 자리**

| | 쓸 수 있는 데이터 |
|---|---|
| `input_dim=4` | 표 형태(2차원)만 |
| `input_shape=(4,)` | 표 형태 |
| `input_shape=(100, 100, 3)` | **이미지처럼 3차원 이상** |

→ 앞으로 이미지를 다루면 `(가로, 세로, 채널)`이 되므로 `input_shape`만 가능

## 4. 스케일링 - 필요한 이유

캘리포니아 주택 데이터의 **실제 값 범위**를 보면 (`keras27`)

```
MedInc          0.500 ~        15.000      <- 소득
HouseAge        1.000 ~        52.000
AveRooms        0.846 ~       141.909
AveBedrms       0.333 ~        34.067
Population      3.000 ~     35682.000      <- 인구
AveOccup        0.692 ~      1243.333
Latitude       32.540 ~        41.950
Longitude    -124.350 ~      -114.310      <- 음수
```

→ 소득은 최대 15인데 인구는 35,682. **2천 배 차이**

→ 모델 입장에서는 `w`를 갱신할 때 **숫자가 큰 특성이 loss를 지배**함. 인구는 조금만 움직여도 loss가 크게 변하고, 소득은 아무리 움직여봐야 묻힘

→ 3일차 §5에서 "특성 스케일이 제각각이면 생기는 문제"라고만 적어둔 것의 해결책

### MinMaxScaler - 0에서 1 사이로

가장 큰 수로 나누기만 하면 최대값은 1이 되지만 **최소값이 0이 되지 않음**

$$x' = \frac{x - \text{MIN}}{\text{MAX} - \text{MIN}}$$

→ MIN을 빼서 시작점을 0으로 맞추고, 폭(MAX-MIN)으로 나눠 1까지 펴는 것

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(x)           # MIN, MAX를 계산 (기준을 잡음)
x = scaler.transform(x) # 그 기준으로 변환
```

→ `fit`은 **기준을 정하는 단계**, `transform`은 **적용하는 단계**. 이 구분이 §5에서 다시 나옴

→ 모든 특성이 0~1이 되면 어느 것도 크기로 우위를 갖지 못함

### 효과 - 모델은 그대로인데 R²가 2.4배

`keras27`은 층 구성·epoch를 하나도 안 바꾸고 스케일링만 넣음

| | 스케일링 전 | 후 |
|---|---|---|
| loss (mse) | 0.9900 | **0.5106** |
| R² | 0.2542 | **0.6154** |
| RMSE | 0.9950 | **0.7146** |

### cf) `1.0000000000000002`

```python
print(np.min(x), np.max(x))     # 0.0 1.0000000000000002
```

→ 1을 넘는 게 아니라 **부동소수점 연산 오차**

→ `(MAX - MIN) / (MAX - MIN)`이 정확히 1로 떨어지지 않는 것. 10진수 소수를 2진수로 저장하면서 생기는 미세한 차이

→ 무시해도 되는 값

## 5. 스케일링 순서 - 분리가 먼저

`keras27`은 순서가 잘못됨. **x 전체를 스케일링한 뒤에 나눔**

```python
# X - 잘못된 순서
scaler.fit(x)                                       # 전체로 기준을 잡음
x = scaler.transform(x)
x_train, x_test, ... = train_test_split(x, y, ...)  # 그 다음에 분리
```

```python
# O - 올바른 순서 (keras28)
x_train, x_test, ... = train_test_split(x, y, ...)  # 1) 먼저 분리

scaler.fit(x_train)                                 # 2) train으로만 기준을 잡고
x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)                   # 3) test는 그 기준으로 변환만
```

→ **`fit`은 train에만. test에는 절대 `fit`하지 않음**

### 데이터 누수(leakage) - 평가가 부풀려짐

전체로 `fit`하면 scaler가 **test의 MIN/MAX까지 알게 됨**

```
전체로 fit    :  scaler가 본 것 = train + test    <- test 정보가 섞임
train으로 fit :  scaler가 본 것 = train만         <- 정상
```

→ test 정보가 훈련 과정에 들어간 것. 5일차 §2의 `casual + registered = count`와 **같은 종류의 문제**

→ 결과적으로 **평가 점수가 실제 성능보다 좋게 나옴**

→ 실전에서는 아직 존재하지도 않는 데이터가 들어오는데, 그 MIN/MAX를 미리 알 방법이 없음

### x_test - 0~1을 벗어나는 건 정상

train 기준으로만 변환하니 **train에서 못 본 범위**는 밖으로 나감

```python
print(np.min(x_train), np.max(x_train))     # 0.0 1.0000000000000002
print(np.min(x_test),  np.max(x_test))      # -0.005005005005005 2.0745532963647566
```

max가 2.07이 나오는 이유를 찾아보면

```
AveOccup 컬럼
  train 의 MIN / MAX  :    0.75 /  599.71
  그 test 행의 원본 값 : 1243.33          <- train 에서 본 적 없는 극단값

  (1243.33 - 0.75) / (599.71 - 0.75) = 2.0746
```

| | 개수 |
|---|---|
| 1 초과 | 1 / 33,024 |
| 0 미만 | 2 / 33,024 |

→ 33,024개 중 **3개뿐**. 이 3개가 train에서 보지 못한 범위의 값

→ 여기서 x_test를 다시 `fit`하면 2.07이 1.0으로 바뀌면서 **이상치라는 사실 자체가 사라짐**

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

d = fetch_california_housing()
x, y = d.data, d.target

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=121)

scaler = MinMaxScaler()
scaler.fit(x_train)                     # train 으로만 fit
x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)       # test 는 transform 만

print(np.min(x_train), np.max(x_train)) # 0.0 1.0000000000000002
print(np.min(x_test),  np.max(x_test))  # -0.005005005005005 2.0745532963647566

# 2.07 이 어느 컬럼인지 찾아보기
i, j = np.unravel_index(np.argmax(x_test), x_test.shape)
print(d.feature_names[j])                           # AveOccup
print(scaler.data_min_[j], scaler.data_max_[j])     # 0.75 599.7142857142857

### cf) `validation_split`에도 같은 문제가 있음

```python
scaler.fit(x_train)                                     # x_train 전체로 fit
model.fit(x_train, y_train, validation_split=0.2)       # 그 x_train에서 val을 떼감
```

→ val도 결국 x_train에서 잘라내므로, **val의 MIN/MAX도 scaler가 이미 알고 있음**

→ 엄밀히 하려면 val을 먼저 떼고(6일차 §3의 `validation_data`) train으로만 `fit`해야 함

→ 실무에서도 보통 이 정도는 넘어가지만, **누수가 없는 건 아니라는 것**은 알고 있을 것